# INV6109 Final — Representation Drift and the Geometry of Innovation

**A self-contained, top-to-bottom pipeline.** Run the cells in order; the last
cells of each track print the deliverable index (PMDI / AIS / PSC) and display the
four required figures inline.

- **Model G** = DistilGPT2 (82M, frozen). **D1/D2** = G + LoRA adapters (shared
  frozen base, so `cos(h_G, h_D)` is well-posed).
- **Drift** is measured at token / hidden / layer-wise levels with cosine
  displacement, neighborhood replacement, and a layer-wise profile.
- Three tracks: **A** Policy vs Market (PMDI), **B** Fiction vs Reality (AIS),
  **C** Paradigm transition (PSC).

> Tip: start with the default (fast) settings to verify everything runs, then
> scale up `CONFIG["train"]["max_steps"]` and `extract` sizes for paper-grade
> numbers.


## 0. Setup

In [ ]:
# If running fresh, install dependencies first (uncomment):
# %pip install torch transformers "peft==0.10.0" nltk scikit-learn umap-learn matplotlib

import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"   # let unsupported MPS ops fall back to CPU
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import re, json, random
from functools import lru_cache
import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, PeftModel, get_peft_model
import nltk
print("torch", torch.__version__)

## 0b. Configuration

Everything that affects results lives here. The defaults are **fast** (so the whole
notebook runs in a few minutes on a laptop). For the figures/numbers you would put
in the paper, raise `train.max_steps` (e.g. 400–1500) and the `extract` sizes.

In [ ]:
CONFIG = {
    "seed": 20260615,
    "base_model": "distilgpt2",          # 82M params -> within the 100M budget
    "device": "auto",                    # "auto" | "cpu" | "cuda" | "mps"
    "lora": {"r": 8, "alpha": 16, "dropout": 0.0,
             "target_modules": ["c_attn", "c_proj", "c_fc"]},
    "train": {"max_steps": 200, "batch_size": 4, "block_size": 160, "lr": 1e-4},
    "extract": {"probe_corpus": "brown", "n_probe_sentences": 30, "max_len": 96,
                "anchor_n_probe": 10, "neighbor_k": 10, "anchor_vocab_size": 200},
    "track_a": {"policy_corpus": "state_union", "market_corpus": "reuters",
                "concepts": ["energy", "security", "trade", "industry",
                             "technology", "investment", "growth", "defense"]},
    "track_b": {"vision_corpus": "gutenberg",
                "vision_files": ["whitman-leaves.txt", "blake-poems.txt", "milton-paradise.txt"],
                "vision_external": [],   # set to real Verne/Wells/Shelley .txt paths to override
                "reality_corpus": "reuters",
                "concepts": ["machine", "power", "flight", "weapon",
                             "energy", "vision", "future", "science"]},
    "track_c": {"corpus": "brown", "domain_a": "science_fiction", "domain_b": "learned",
                "probe_corpus": "reuters",   # neutral probe (NOT brown, the adapted corpus)
                "concepts": ["time", "space", "matter", "mind",
                             "method", "truth", "force", "system"]},
}

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)

def resolve_device(spec):
    if spec != "auto":
        return torch.device(spec)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

set_seed(CONFIG["seed"])
DEVICE = resolve_device(CONFIG["device"])
for d in ("artifacts", "results", "figures"):
    os.makedirs(d, exist_ok=True)
print("device:", DEVICE)

## 1. Data (NLTK corpora, all English)

In [ ]:
_REQUIRED = {"state_union": "corpora/state_union", "reuters": "corpora/reuters",
             "gutenberg": "corpora/gutenberg", "brown": "corpora/brown",
             "punkt": "tokenizers/punkt", "punkt_tab": "tokenizers/punkt_tab"}

def ensure_nltk():
    for pkg, path in _REQUIRED.items():
        try:
            nltk.data.find(path)
        except LookupError:
            nltk.download(pkg, quiet=True)

def _clean(sent):
    return re.sub(r"\s+", " ", " ".join(sent)).strip()

@lru_cache(maxsize=None)
def corpus_sentences(name, category=None, fileids=None):
    ensure_nltk()
    corpus = getattr(nltk.corpus, name)
    kwargs = {}
    if category is not None:
        kwargs["categories"] = category
    if fileids is not None:
        kwargs["fileids"] = list(fileids)
    sents = [_clean(s) for s in corpus.sents(**kwargs)]
    return tuple(s for s in sents if len(s.split()) >= 4)

def corpus_as_text_blocks(name, category=None, fileids=None, max_chars=None):
    text = "\n".join(corpus_sentences(name, category=category, fileids=fileids))
    return text[:max_chars] if max_chars else text

def sentences_containing(concept, sentences, n, seed):
    pat = re.compile(rf"\b{re.escape(concept)}\b", re.IGNORECASE)
    hits = [s for s in sentences if pat.search(s)]
    rng = random.Random(seed); rng.shuffle(hits)
    return hits[:n]

_MINI_STOP = {"the","and","for","that","with","this","from","are","was","has",
              "have","not","but","they","his","her","its","our"}
def _stopwords():
    try:
        nltk.data.find("corpora/stopwords")
    except LookupError:
        try:
            nltk.download("stopwords", quiet=True)
        except Exception:
            return _MINI_STOP
    return set(nltk.corpus.stopwords.words("english"))

def build_anchor_vocab(sentences, size, must_include=()):
    stop = _stopwords()
    freq = {}
    for s in sentences:
        for tok in re.findall(r"[a-zA-Z]+", s.lower()):
            if len(tok) > 2 and tok not in stop:
                freq[tok] = freq.get(tok, 0) + 1
    ranked = [w for w, _ in sorted(freq.items(), key=lambda kv: -kv[1])]
    vocab = list(dict.fromkeys(list(must_include) + ranked))
    return vocab[:max(size, len(must_include))]

# --- Track B Path 2: external real Verne/Wells/Shelley (optional) ---
_GUT_START = re.compile(r"\*\*\*\s*START OF (?:THE|THIS) PROJECT GUTENBERG.*?\*\*\*", re.I | re.S)
_GUT_END   = re.compile(r"\*\*\*\s*END OF (?:THE|THIS) PROJECT GUTENBERG.*?\*\*\*", re.I | re.S)

def external_text_blocks(paths, max_chars=None):
    ensure_nltk()
    from nltk.tokenize import sent_tokenize
    out = []
    for p in paths:
        with open(p, "r", encoding="utf-8", errors="ignore") as fh:
            raw = fh.read()
        m = _GUT_START.search(raw); start = m.end() if m else 0
        m = _GUT_END.search(raw, start); end = m.start() if m else len(raw)
        for sent in sent_tokenize(raw[start:end]):
            cln = re.sub(r"\s+", " ", sent).strip()
            if len(cln.split()) >= 4:
                out.append(cln)
    text = "\n".join(out)
    return text[:max_chars] if max_chars else text

ensure_nltk()
print("NLTK corpora ready")

## 2. Model G (frozen) and LoRA domain models

In [ ]:
def load_tokenizer():
    tok = AutoTokenizer.from_pretrained(CONFIG["base_model"])
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok

def load_general_model():
    model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"]).to(DEVICE).eval()
    for p in model.parameters():
        p.requires_grad_(False)
    return model

def make_lora_model():
    lc = CONFIG["lora"]
    model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"])
    lora = LoraConfig(r=lc["r"], lora_alpha=lc["alpha"], lora_dropout=lc["dropout"],
                      target_modules=lc["target_modules"], task_type="CAUSAL_LM", bias="none")
    return get_peft_model(model, lora).to(DEVICE)

def load_domain_model(adapter_dir):
    base = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"])
    model = PeftModel.from_pretrained(base, adapter_dir).to(DEVICE).eval()
    for p in model.parameters():
        p.requires_grad_(False)
    return model

def train_domain_adapter(text, out_dir):
    from torch.utils.data import DataLoader, TensorDataset
    set_seed(CONFIG["seed"])
    model = make_lora_model(); model.train()
    bs = CONFIG["train"]["block_size"]
    ids = TOK(text, return_tensors="pt").input_ids[0]
    n = (ids.numel() // bs) * bs
    if n == 0:
        ids = torch.nn.functional.pad(ids, (0, bs - ids.numel()), value=TOK.pad_token_id); n = bs
    ds = TensorDataset(ids[:n].view(-1, bs))
    g = torch.Generator().manual_seed(CONFIG["seed"])
    dl = DataLoader(ds, batch_size=CONFIG["train"]["batch_size"], shuffle=True, generator=g)
    trainable = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(trainable, lr=CONFIG["train"]["lr"])
    losses, step, max_steps = [], 0, CONFIG["train"]["max_steps"]
    while step < max_steps:
        for (batch,) in dl:
            batch = batch.to(DEVICE)
            out = model(input_ids=batch, labels=batch)
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)   # <-- prevents divergence to NaN
            opt.step(); opt.zero_grad()
            losses.append(float(out.loss.detach().cpu())); step += 1
            if step >= max_steps:
                break
    os.makedirs(out_dir, exist_ok=True)
    model.save_pretrained(out_dir)
    print(f"  [{os.path.basename(out_dir)}] steps={step} loss {losses[0]:.3f} -> {losses[-1]:.3f}")
    return out_dir

def adapt_or_load(name, text):
    out = os.path.join("artifacts", name)
    if os.path.exists(os.path.join(out, "adapter_config.json")):
        print(f"  [{name}] cached")
        return out
    return train_domain_adapter(text, out)

# Load the shared base once.
TOK = load_tokenizer()
G = load_general_model()
print("Model G loaded:", CONFIG["base_model"])

## 3. Concept representation at three levels

`h_M(c)` = mean over a fixed, **shared** probe set of the mean-pooled hidden state at
`c`'s sub-token positions. Sharing the probe across G/D1/D2 isolates *adaptation*
drift from *context* drift.

In [ ]:
def token_embedding(model, concept):
    emb = model.get_input_embeddings().weight.detach()
    ids = TOK(" " + concept, add_special_tokens=False).input_ids
    return emb[ids].mean(dim=0).cpu().float().numpy()

@torch.no_grad()
def contextual_reps(model, sentences, concept):
    pat = re.compile(rf"\b{re.escape(concept)}\b", re.IGNORECASE)
    per_layer = {}
    max_len = CONFIG["extract"]["max_len"]
    for sent in sentences:
        enc = TOK(sent, return_offsets_mapping=True, truncation=True,
                  max_length=max_len, return_tensors="pt")
        offsets = enc.pop("offset_mapping")[0].tolist()
        spans = [(m.start(), m.end()) for m in pat.finditer(sent)]
        idx = [i for i, (a, b) in enumerate(offsets)
               if a != b and any(a < e and s < b for s, e in spans)]
        if not idx:
            continue
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        out = model(**enc, output_hidden_states=True)
        sel = torch.tensor(idx, device=DEVICE)
        for layer, hs in enumerate(out.hidden_states):
            pooled = hs[0].index_select(0, sel).mean(dim=0)
            per_layer.setdefault(layer, []).append(pooled.cpu().float().numpy())
    return {l: np.mean(v, axis=0) for l, v in per_layer.items() if v}

def anchor_matrix(model, anchors, sentences, layer):
    ex = CONFIG["extract"]
    rows = []
    for i, w in enumerate(anchors):
        probe = sentences_containing(w, sentences, ex["anchor_n_probe"], CONFIG["seed"] + i)
        reps = contextual_reps(model, probe, w) if probe else {}
        rows.append(reps[layer] if layer in reps else token_embedding(model, w))
    return np.vstack(rows)

## 4. Drift metrics

In [ ]:
def cosine(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return 0.0 if na == 0 or nb == 0 else float(np.dot(a, b) / (na * nb))

def cosine_displacement(a, b):
    return 1.0 - cosine(a, b)

def layer_profile(r1, r2):
    return {l: cosine_displacement(r1[l], r2[l]) for l in sorted(set(r1) & set(r2))}

def _nearest(idx, mat, k):
    v = mat[idx]
    sims = (mat @ v) / (np.linalg.norm(mat, axis=1) * (np.linalg.norm(v) + 1e-12) + 1e-12)
    sims[idx] = -np.inf
    return set(np.argsort(-sims)[:k])

def neighborhood_replacement_rate(m1, m2, idx, k):
    n1, n2 = _nearest(idx, m1, k), _nearest(idx, m2, k)
    union = len(n1 | n2)
    return 1.0 - (len(n1 & n2) / union if union else 1.0)

def aggregate(values):
    arr = np.asarray([v for v in values if v == v], dtype=float)
    if arr.size == 0:
        return {"mean": float("nan"), "std": float("nan")}
    return {"mean": float(arr.mean()), "std": float(arr.std()),
            "min": float(arr.min()), "max": float(arr.max())}

def drift_report(concept, ra, rb, ea, eb, nrr, probe_layer):
    prof = layer_profile(ra, rb)
    return {"concept": concept,
            "token_embedding_drift": cosine_displacement(ea, eb),
            "hidden_state_drift": prof.get(probe_layer, float("nan")),
            "layer_profile": {str(l): v for l, v in prof.items()},
            "peak_layer": (max(prof, key=prof.get) if prof else None),
            "neighborhood_replacement_rate": nrr}

## 5. Deliverable indices (PMDI / AIS / PSC)

In [ ]:
def pmdi(per_concept, w_cos=0.6, w_nrr=0.4):
    ranked = [{"concept": r["concept"],
               "pmdi": w_cos * r["hidden_state_drift"] + w_nrr * r["neighborhood_replacement_rate"],
               "hidden_state_drift": r["hidden_state_drift"],
               "neighborhood_replacement_rate": r["neighborhood_replacement_rate"]}
              for r in per_concept]
    ranked.sort(key=lambda x: -x["pmdi"])
    return {"per_concept": ranked, "corpus_pmdi": aggregate([r["pmdi"] for r in ranked])["mean"]}

def ais(per_concept):
    aligns = [{"concept": r["concept"], "ais": 1.0 - r["hidden_state_drift"],
               "neighborhood_replacement_rate": r["neighborhood_replacement_rate"]}
              for r in per_concept]
    stats = aggregate([x["ais"] for x in aligns]); thr = stats["mean"] + stats["std"]
    for x in aligns:
        x["precursor"] = bool(x["ais"] >= thr)
    aligns.sort(key=lambda x: -x["ais"])
    return {"per_concept": aligns, "precursor_threshold": thr, "corpus_ais": stats["mean"]}

def psc(per_concept, centroid_shift, silhouette_gain):
    mean_drift = aggregate([r["hidden_state_drift"] for r in per_concept])["mean"]
    coef = float(np.mean([mean_drift, centroid_shift, max(silhouette_gain, 0.0)]))
    ranked = sorted(per_concept, key=lambda r: -r["hidden_state_drift"])
    return {"psc": coef,
            "components": {"mean_concept_drift": mean_drift,
                           "centroid_shift": centroid_shift, "silhouette_gain": silhouette_gain},
            "top_moving_concepts": [r["concept"] for r in ranked[:5]]}

## 6. Visualization (the four required figures, shown inline)

In [ ]:
def _finish(fig, path):
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()

def umap_map(mats, labels, path, title, highlight=()):
    import umap
    names = list(mats.keys())
    stacked = np.nan_to_num(np.vstack([mats[n] for n in names]))
    nn = max(2, min(15, stacked.shape[0] - 1))
    emb = umap.UMAP(n_neighbors=nn, min_dist=0.1, random_state=CONFIG["seed"],
                    metric="cosine").fit_transform(stacked)
    fig, ax = plt.subplots(figsize=(8, 6)); rows = mats[names[0]].shape[0]; hl = set(highlight)
    for i, name in enumerate(names):
        chunk = emb[i*rows:(i+1)*rows]
        ax.scatter(chunk[:, 0], chunk[:, 1], s=14, alpha=0.5, label=name)
        for j, lab in enumerate(labels):
            if lab in hl:
                ax.annotate(lab, (chunk[j, 0], chunk[j, 1]), fontsize=8)
    ax.set_title(title); ax.legend(); _finish(fig, path)

def drift_heatmap(concepts, layers, matrix, path, title):
    fig, ax = plt.subplots(figsize=(1 + 0.5*len(layers), 0.5*len(concepts) + 1))
    im = ax.imshow(matrix, aspect="auto", cmap="magma")
    ax.set_xticks(range(len(layers))); ax.set_xticklabels([f"L{l}" for l in layers])
    ax.set_yticks(range(len(concepts))); ax.set_yticklabels(concepts)
    ax.set_xlabel("layer"); ax.set_title(title); fig.colorbar(im, ax=ax, label="1 - cosine")
    _finish(fig, path)

def layerwise_displacement(profiles, path, title):
    fig, ax = plt.subplots(figsize=(8, 5))
    for concept, prof in profiles.items():
        ls = sorted(prof); ax.plot(ls, [prof[l] for l in ls], marker="o", label=concept)
    ax.set_xlabel("layer"); ax.set_ylabel("drift (1 - cosine)"); ax.set_title(title)
    ax.legend(fontsize=8, ncol=2); _finish(fig, path)

def trajectory_diagram(pg, pd, labels, path, title):
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(pg[:, 0], pg[:, 1], c="steelblue", s=30, label="General (G)")
    ax.scatter(pd[:, 0], pd[:, 1], c="crimson", s=30, label="Domain (D)")
    for i, lab in enumerate(labels):
        ax.annotate("", xy=pd[i], xytext=pg[i],
                    arrowprops=dict(arrowstyle="->", color="gray", alpha=0.7))
        ax.annotate(lab, pd[i], fontsize=8)
    ax.set_title(title); ax.legend(); _finish(fig, path)

def pca_2d(mat):
    from sklearn.decomposition import PCA
    return PCA(n_components=2, random_state=CONFIG["seed"]).fit_transform(mat)

## 7. Pipeline helpers (adapt -> extract -> measure -> plot)

In [ ]:
def build_probe_map(concepts, probe_sents):
    n = CONFIG["extract"]["n_probe_sentences"]
    return {c: sentences_containing(c, probe_sents, n, CONFIG["seed"] + i)
            for i, c in enumerate(concepts)}

def concept_reps(model, probe_map):
    return {c: contextual_reps(model, sents, c) for c, sents in probe_map.items()}

def last_layer(reps_by_concept):
    layers = set()
    for r in reps_by_concept.values():
        layers |= set(r)
    return max(layers) if layers else 0

def measure_pair(concepts, reps_a, reps_b, model_a, model_b, anchors, probe_sents, probe_layer):
    k = CONFIG["extract"]["neighbor_k"]
    mat_a = anchor_matrix(model_a, anchors, probe_sents, probe_layer)
    mat_b = anchor_matrix(model_b, anchors, probe_sents, probe_layer)
    aidx = {w: i for i, w in enumerate(anchors)}
    reports = []
    for c in concepts:
        ra, rb = reps_a.get(c, {}), reps_b.get(c, {})
        if not ra or not rb:
            continue
        ea, eb = token_embedding(model_a, c), token_embedding(model_b, c)
        nrr = neighborhood_replacement_rate(mat_a, mat_b, aidx[c], k) if c in aidx else float("nan")
        reports.append(drift_report(c, ra, rb, ea, eb, nrr, probe_layer))
    return reports, mat_a, mat_b

def make_core_figures(reports, mat_a, mat_b, anchors, concepts, la, lb, tag, probe_layer):
    umap_map({la: mat_a, lb: mat_b}, anchors, f"figures/{tag}_umap.png",
             f"{tag}: UMAP representation map", highlight=concepts)
    layers = sorted({int(l) for r in reports for l in r["layer_profile"]})
    heat = np.nan_to_num(np.array([[r["layer_profile"].get(str(l), np.nan) for l in layers]
                                   for r in reports]))
    drift_heatmap([r["concept"] for r in reports], layers, heat,
                  f"figures/{tag}_heatmap.png", f"{tag}: drift heatmap")
    profiles = {r["concept"]: {int(l): v for l, v in r["layer_profile"].items()} for r in reports}
    layerwise_displacement(profiles, f"figures/{tag}_layerwise.png", f"{tag}: layer-wise displacement")
    idx = [anchors.index(c) for c in concepts if c in anchors]
    if idx:
        joint = pca_2d(np.vstack([mat_a[idx], mat_b[idx]])); h = len(idx)
        trajectory_diagram(joint[:h], joint[h:], [anchors[i] for i in idx],
                           f"figures/{tag}_trajectory.png",
                           f"{tag}: innovation trajectory ({la} -> {lb})")

## 8. Track A — Policy vs. Market (PMDI)

Policy (`state_union`) -> D1, Market (`reuters`) -> D2, neutral probe = `brown`.
Also computes the frozen-G **context-drift control** to separate adaptation from
context.

In [ ]:
ta = CONFIG["track_a"]
policy_text = corpus_as_text_blocks(ta["policy_corpus"])
market_text = corpus_as_text_blocks(ta["market_corpus"], max_chars=len(policy_text))
print("Training Track A adapters...")
D1 = load_domain_model(adapt_or_load("trackA_policy", policy_text))
D2 = load_domain_model(adapt_or_load("trackA_market", market_text))

probe = corpus_sentences(CONFIG["extract"]["probe_corpus"])
concepts = ta["concepts"]
anchors = build_anchor_vocab(probe, CONFIG["extract"]["anchor_vocab_size"], must_include=concepts)
probe_map = build_probe_map(concepts, probe)
reps_d1, reps_d2, reps_g = concept_reps(D1, probe_map), concept_reps(D2, probe_map), concept_reps(G, probe_map)
probe_layer = last_layer(reps_d1)

reports, mat_d1, mat_d2 = measure_pair(concepts, reps_d1, reps_d2, D1, D2, anchors, probe, probe_layer)
A_index = pmdi(reports)

# Adaptation drift away from G, and the frozen-G context-drift control.
adapt = aggregate([layer_profile(reps_g[c], reps_d1[c]).get(probe_layer)
                   for c in concepts if reps_g.get(c) and reps_d1.get(c)])
pol, mkt = corpus_sentences(ta["policy_corpus"]), corpus_sentences(ta["market_corpus"])
ctx = []
for i, c in enumerate(concepts):
    ps = sentences_containing(c, pol, CONFIG["extract"]["n_probe_sentences"], CONFIG["seed"] + i)
    ms = sentences_containing(c, mkt, CONFIG["extract"]["n_probe_sentences"], CONFIG["seed"] + i)
    rp, rm = (contextual_reps(G, ps, c) if ps else {}), (contextual_reps(G, ms, c) if ms else {})
    if rp and rm:
        ctx.append(layer_profile(rp, rm).get(probe_layer))
context_drift = aggregate(ctx)

print(f"\nCorpus PMDI = {A_index['corpus_pmdi']:.4f}")
print("Top divergent concepts:", [r["concept"] for r in A_index["per_concept"][:3]])
print(f"Adaptation drift G->D1 = {adapt['mean']:.4f}  vs  context-drift control = {context_drift['mean']:.4f}")
json.dump({"PMDI": A_index, "adaptation_drift": adapt, "context_drift": context_drift,
           "per_concept": reports}, open("results/track_a.json", "w"), indent=2)
make_core_figures(reports, mat_d1, mat_d2, anchors, concepts, "policy(D1)", "market(D2)", "trackA", probe_layer)

## 9. Track B — Fiction vs. Reality (AIS)

Vision (justified `gutenberg` subset — or external real Verne/Wells/Shelley via
`vision_external`) -> D_vision; `reuters` -> D_reality.

In [ ]:
tb = CONFIG["track_b"]
if tb["vision_external"]:
    vision_text = external_text_blocks(tb["vision_external"]); vsrc = f"external:{tb['vision_external']}"
else:
    vision_text = corpus_as_text_blocks(tb["vision_corpus"], fileids=tuple(tb["vision_files"]))
    vsrc = f"nltk.gutenberg:{tb['vision_files']}"
reality_text = corpus_as_text_blocks(tb["reality_corpus"], max_chars=len(vision_text))
print("vision source =", vsrc)
DV = load_domain_model(adapt_or_load("trackB_vision", vision_text))
DR = load_domain_model(adapt_or_load("trackB_reality", reality_text))

probe = corpus_sentences(CONFIG["extract"]["probe_corpus"])
concepts = tb["concepts"]
anchors = build_anchor_vocab(probe, CONFIG["extract"]["anchor_vocab_size"], must_include=concepts)
probe_map = build_probe_map(concepts, probe)
reps_v, reps_r = concept_reps(DV, probe_map), concept_reps(DR, probe_map)
probe_layer = last_layer(reps_v)
reports, mat_v, mat_r = measure_pair(concepts, reps_v, reps_r, DV, DR, anchors, probe, probe_layer)
B_index = ais(reports)

print(f"\nCorpus AIS = {B_index['corpus_ais']:.4f}")
print("Precursor candidates:", [x["concept"] for x in B_index["per_concept"] if x["precursor"]])
print("Caveat: AIS is alignment, not temporal causation (corpora are undated).")
json.dump({"AIS": B_index, "vision_source": vsrc, "per_concept": reports},
          open("results/track_b.json", "w"), indent=2)
make_core_figures(reports, mat_v, mat_r, anchors, concepts, "vision", "reality", "trackB", probe_layer)

## 10. Track C — Scientific Paradigm Transition (PSC)

Two contrasting Brown categories adapted; neutral probe = `reuters`.

In [ ]:
from sklearn.metrics import silhouette_score
tc = CONFIG["track_c"]
text_a = corpus_as_text_blocks(tc["corpus"], category=tc["domain_a"])
text_b = corpus_as_text_blocks(tc["corpus"], category=tc["domain_b"], max_chars=len(text_a))
DA = load_domain_model(adapt_or_load(f"trackC_{tc['domain_a']}", text_a))
DB = load_domain_model(adapt_or_load(f"trackC_{tc['domain_b']}", text_b))

probe = corpus_sentences(tc["probe_corpus"])   # neutral, NOT brown
concepts = tc["concepts"]
anchors = build_anchor_vocab(probe, CONFIG["extract"]["anchor_vocab_size"], must_include=concepts)
probe_map = build_probe_map(concepts, probe)
reps_a, reps_b = concept_reps(DA, probe_map), concept_reps(DB, probe_map)
probe_layer = last_layer(reps_a)
reports, mat_a, mat_b = measure_pair(concepts, reps_a, reps_b, DA, DB, anchors, probe, probe_layer)

centroid_shift = cosine_displacement(mat_a.mean(axis=0), mat_b.mean(axis=0))
stack = np.vstack([mat_a, mat_b]); labels = np.array([0]*len(mat_a) + [1]*len(mat_b))
try:
    sil = float(silhouette_score(stack, labels, metric="cosine"))
except Exception:
    sil = 0.0
C_index = psc(reports, centroid_shift, sil)

print(f"\nPSC = {C_index['psc']:.4f}  (centroid_shift={centroid_shift:.4f}, silhouette={sil:.4f})")
print("Top moving concepts:", C_index["top_moving_concepts"][:3])
json.dump({"PSC": C_index, "domains": [tc["domain_a"], tc["domain_b"]], "per_concept": reports},
          open("results/track_c.json", "w"), indent=2)
make_core_figures(reports, mat_a, mat_b, anchors, concepts, tc["domain_a"], tc["domain_b"], "trackC", probe_layer)

## 11. Summary

In [ ]:
print("=== Deliverable indices ===")
print(f"Track A  PMDI = {A_index['corpus_pmdi']:.4f}")
print(f"Track B  AIS  = {B_index['corpus_ais']:.4f}")
print(f"Track C  PSC  = {C_index['psc']:.4f}")
print("\nResults JSON -> results/  |  Figures -> figures/ (also shown inline above)")
print("To get paper-grade numbers, raise CONFIG['train']['max_steps'] and the extract sizes, then re-run.")